# Review notes — Gold customer dimension

This inherited notebook is a learning example, not a verified production SCD Type 2 implementation. Its original logic does not reliably detect changed attributes, close prior validity records, maintain a current-row flag, or guarantee stable surrogate keys. Review `docs/scd-type2-explanation.md` before describing it as SCD Type 2. This notebook has not been executed after editing.


Loading Libraries

In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
dbutils.widgets.text("init_load_flag", "0")
init_load_flag = int(dbutils.widgets.get("init_load_flag"))

#### Data Reading

## SOURCE FILE -> Silver_Customers

In [ ]:
df = spark.sql("select * from databricks_etl_catalog.silver.customer_silver")
df.display()

### Removing Duplicates based on Primary key.
#### Primary Key: customer_id (unique key)

In [ ]:
df = df.dropDuplicates(subset=['customer_id'])
df.limit(10).display()

## **Dividing Old Vs. New Records**

In [ ]:
if init_load_flag == 0:
    df_old = spark.sql('''select DimCustomerKey, customer_id, create_date, update_date 
                       from databricks_etl_catalog.gold.DimCustomers''')  # assuming this table (DimCustomers) exists, it will be created below.

else:
    df_old = spark.sql('''select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date 
                       from databricks_etl_catalog.silver.customer_silver where 1=0''')  # where 1=0; this only returns the column from schema. And we are putting 0 values to pseudo columns as they are not present in silver table


In [ ]:
df_old.display()

#### Renaming Columns of Old Records

In [ ]:
df_old = df_old.withColumnRenamed("DimCustomerKey", "old_DimCustomerKey")\
                .withColumnRenamed("customer_id", "old_customer_id")\
                .withColumnRenamed("create_date", "old_create_date")\
                .withColumnRenamed("update_date", "old_update_date")

## **Applying Join with Old Records**

In [ ]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')
df_join.display()

## **Seperating New Vs. Old Records**

In [ ]:
df_new = df_join.filter(df_join['old_DimCustomerKey'].isNull())    # where all records have null values for old_DimCustomerKey
df_new.limit(10).display()

In [ ]:
df_old = df_join.filter(df_join.old_DimCustomerKey.isNotNull())    # as this df_old is already added into Gold Layer 
df_old.limit(10).display()

## **Preparing df_old Record**

In [ ]:
# Dropping all columns that aren't required
df_old = df_old.drop('old_customer_id', 'old_update_date')

# Renaming 'Old_DimCustomerKey' to 'DimCustomerKey'
df_old = df_old.withColumnRenamed("old_DimCustomerKey", "DimCustomerKey")

# Renaming old_create_date to create_date
df_old = df_old.withColumnRenamed("old_create_date", "create_date") 
df_old = df_old.withColumn("create_date", to_timestamp(col("create_date")))

# Recreating 'update_date' column with current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())


In [ ]:
df_old.display()

## **Preparing df_new Records**

In [ ]:
df_new.limit(10).display()

##### Removing columns like : old_DimCustomerKey, old_customer_id, old_create_date, old_update_date: because all having 'null' values. Creating new columns instead

In df_new, the column 'create_date' will have current_timestamp function to retrieve current date & time. As df_new are processed now unlike df_old which are already in gold layer existing.

In [ ]:
# Dropping unnecessary columns that are not required now
df_new = df_new.drop('old_DimCustomerKey', 'old_customer_id', 'old_create_date', 'old_update_date')

# Recreting new columns "update_date" & "current_date" by using function current_timestamp()
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())

df_new.display()


#### Assigning surrogate key column to Data Frame

Surrogate Key: a pseudo/fake key that create in dimension & can easily apply joins between fact and dimension tables

Dim_Key: a surrogate key of our primary key [customer_id]

## **Surrogate Key - For 1**

Assigning dim surrogate key column to df_new

In [ ]:
# Creating surrogate keys for primary keys
df_new = df_new.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))   
df_new.display()


## **Adding Max Surrogate Key**

In [ ]:
if init_load_flag == 1:
    max_surrogate_key = 0  # max surrogate key assigning to 0

else:
  df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricks_etl_catalog.gold.DimCustomers")
  # Converting df_maxsk to a list (max_sk)
  max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']   # here with collect function we can convert values (collected dataframe as df_maxsk) to a list
  

Adding surrogate keys monotronically as increasing in df_new now one by one.

In [ ]:
# Adding the max_sk to the new records with increments
df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key)+col("DimCustomerKey"))  # For ex. 2000+1 , 2000+2, and so on.

## **Union of df_old & df_new**

In [ ]:
df_final = df_new.unionByName(df_old)

In [ ]:
df_final.display()

## **Upsert Table = Upload + Insert**

### SCD Type 1 : Overwrite

In [ ]:
from delta.tables import DeltaTable

### **Merging files**

In [ ]:

if spark.catalog.tableExists("databricks_etl_catalog.gold.DimCustomers"):
        # Saving data to gold folder created in Azure Data Lake Storage Gen2
    dlt_obj = DeltaTable.forPath(spark, f"abfss://gold@{storage_account}.dfs.core.windows.net/DimCustomers")

    # Now merging & upsert the df_final to a target table
    dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.DimCustomerKey = src.DimCustomerKey")\
                        .whenMatchedUpdateAll()\
                        .whenNotMatchedInsertAll()\
                        .execute()
        

else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path", f"abfss://gold@{storage_account}.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("databricks_etl_catalog.gold.DimCustomers")


In [ ]:
%sql
SELECT *
FROM databricks_etl_catalog.gold.dimcustomers